This notebook recieves a json with all the annotationes and returns a csv by baby with their time serie

In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import pandas as pd

BASE_PATH=r"D:\newbornAligned"

with open("json/final_dataset_annotated_zone_v2.json", "r") as file:
    data=json.load(file)

In [4]:
column_order = ["Zona1", "codo_izq", "codo_drc", "rodilla_izq", "rodilla_drc", "mano_izq", "mano_drc", "pie_izq", "pie_drc"]

def extract_baby_id(path):
    return int(path[:2])

def extract_timestamp(path):
    filename=path.split("\\")[-1]
    base=filename.replace("HM", "").replace(".jpeg", "")
    return datetime.strptime(base, "%Y%m%d%H%M%S")

records=[]

for entry in data:
    baby_id=extract_baby_id(entry["thermal_image"])
    timestamp=extract_timestamp(entry["thermal_image"])
    for anot in entry["anotaciones"]:
        if anot["class"]!="no_present":
            records.append({"baby_id": baby_id, "timestamp": timestamp, "point": anot["class"], "temperature": anot["temperature"]})

df=pd.DataFrame(records)
df.sort_values(by=["baby_id", "timestamp"], inplace=True)

os.makedirs("csv_by_baby_filtered", exist_ok=True)

for baby_id, group in df.groupby("baby_id"):
    pivot_table=group.pivot_table(index="timestamp", columns="point", values="temperature", aggfunc="mean")

    ordered_cols=[col for col in column_order]
    pivot_table=pivot_table.reindex(columns=ordered_cols)

    pivot_table.reset_index(inplace=True)

    final_columns=["timestamp"]+ordered_cols
    pivot_table=pivot_table[final_columns]

    filename=f"timeserie_baby_{baby_id}.csv"
    pivot_table.to_csv(filename, index=False)